In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import contextlib
import io

from models.cdiff_lightning import LitCfgDiffusion
from models.cvae_lightning import LitVAE
from models.cbegan_lightning import SpectralCBEGAN

from utils.utils import encode_labels

In [2]:
# Load data info, scalers and the categorical encoder
label_encoder = joblib.load("../pretrained_models/scalers/label_encoder.pkl")
minmax_scaler = joblib.load("../pretrained_models/scalers/minmax_scaler.pkl")
categorical_conditions = joblib.load("../pretrained_models/scalers/categorical_conditions.pkl")
continuous_conditions = joblib.load("../pretrained_models/scalers/continuous_conditions.pkl")
standard_scaler = joblib.load("../pretrained_models/scalers/std_scaler.pkl")

/u/nkerschb/.local/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/u/nkerschb/.local/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/u/nkerschb/.local/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.6.0. This might lead 

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# cbegan
cbegan = SpectralCBEGAN.load_from_checkpoint("../pretrained_models/weights_cbegan.ckpt", map_location=device)
cbegan.to(device)
cbegan.eval()

# cvae
checkpoint = torch.load(
    "../pretrained_models/weights_cvae.ckpt",
    weights_only=False,
    map_location=device
)

with contextlib.redirect_stdout(io.StringIO()):
    hparams = checkpoint['hyper_parameters']
    cvae = LitVAE(**hparams)
    cvae.load_state_dict(checkpoint['state_dict'])
    cvae.to(device)
    cvae.eval()

cdiff = LitCfgDiffusion.load_from_checkpoint("../cdiff_logs/run/version_8/checkpoints/cdiff-epoch=482.ckpt")
cdiff.to(device)
cdiff.eval()

LitCfgDiffusion(
  (cdiff): CfgDiffusion(
    (time_embed): Sequential(
      (0): SinusoidalEmbedding()
      (1): Linear(in_features=128, out_features=128, bias=True)
      (2): SiLU()
    )
    (cond_proj): Sequential(
      (0): Linear(in_features=60, out_features=128, bias=True)
      (1): SiLU()
      (2): Linear(in_features=128, out_features=128, bias=True)
    )
    (net): Sequential(
      (0): Linear(in_features=775, out_features=1024, bias=True)
      (1): ResMLPBlock(
        (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): SiLU()
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (2): ResMLPBlock(
        (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): SiLU()
        (fc2): Linear(in_features=4096

In [5]:
models = {
    'CVAE': cvae,
    'CBEGAN': cbegan,
    'CDIFF': cdiff,
}

In [6]:
def batch_scale_and_encode_conditions(input_dicts, device):
    """ """
    cat_vals = []
    cont_vals = []

    for input_dict in input_dicts:
        # Categorical
        cat = [label_encoder.transform([input_dict[col]])[0] for col in categorical_conditions]
        cat_vals.append(cat)

        # Continuous
        cont_df = pd.DataFrame([input_dict], columns=continuous_conditions)
        cont = minmax_scaler.transform(cont_df)[0]
        cont_vals.append(cont)

    # Stack and convert to tensors
    cat_tensor = torch.tensor(np.array(cat_vals), dtype=torch.float32, device=device)
    cont_tensor = torch.tensor(np.array(cont_vals), dtype=torch.float32, device=device)
    
    label_dict = {}
    for i, col in enumerate(categorical_conditions):
        label_dict[col] = cat_tensor[:, i].unsqueeze(1)
    for i, col in enumerate(continuous_conditions):
        label_dict[col] = cont_tensor[:, i].unsqueeze(1)

    return label_dict


def generate_spectra(model, conditions, latent_dim, embedding_dim, device):
    """ """
    if isinstance(conditions, dict):
        conditions = [conditions]
    N = len(conditions)
        
    noise = torch.randn(N, latent_dim).to(device)
    condition_dict = batch_scale_and_encode_conditions(conditions, device)
    encoded_conditions = encode_labels(condition_dict, embedding_dim, device=device)

    # 3. Generate spectra
    with torch.no_grad():
        if hasattr(model, "generator"):  # CBEGAN
            spectra = model.generator(noise, encoded_conditions)
        elif hasattr(model, "vae"):  # CVAE
            spectra = model.vae.decode(noise, condition_dict)
        elif hasattr(model, "cdiff"):  # CDIFF
            spectra = model.generate_with_condition_dict(condition_dict, sampling_method='ddpm')
        else:
            raise ValueError("Model type not recognized.")

    return spectra.cpu().numpy()

In [7]:
from dataset.data_classes import PytorchSpectraDataset

In [8]:
from dataset.dataset_builder import SpectralDatasetBuilder
dataset_builder = SpectralDatasetBuilder(data_path='../data/in-silico-gen-data.parquet', data_split={"train": 0.8, "test": 0.2}, config_path='../config.yaml')

dataset = PytorchSpectraDataset("../data/h4h_raw_2024-09-16_with_subject_ids.parquet")

In [9]:
y_real = dataset.df_test_normalized[['age', 'sex', 'bmi']]
bmi_age = y_real[['bmi', 'age']]
bmi_age = dataset_builder.minmax_scaler.inverse_transform(bmi_age).T
bmi = bmi_age[0]
age = bmi_age[1]
sex = y_real['sex'].map({0: 'male', 1:'female'}).values

In [10]:
y_df = pd.DataFrame({
    "age": age,
    "sex": sex,
    "bmi": bmi
})

y = y_df.to_dict(orient="records")

In [11]:
synthetic_datasets = {}

for model_name, model in models.items():
    print(f'now generating {model_name}')
    if model_name == 'CVAE':
        latent_dim = 100
    elif model_name == 'CBEGAN':
        latent_dim = 64
    else:
        latent_dim = 128
        
    X = generate_spectra(model, y, latent_dim=latent_dim, embedding_dim=20, device=device)
    X = pd.DataFrame(X, columns=dataset_builder.features)
    X[['age', 'sex', 'bmi']] = pd.DataFrame(y)
    X['test_set'] = dataset.df_test_normalized['test_set'].values
    X.to_parquet(f'../data/{model_name}_generated.parquet')
    
    synthetic_datasets[model_name] = X 

now generating CVAE
now generating CBEGAN
now generating CDIFF


In [12]:
synthetic_datasets

{'CVAE':       1000.88098144531  1002.80944824219  1004.73791503906  1006.66638183594  \
 0            -0.603065         -0.604608         -0.596331         -0.598482   
 1             0.844790          0.846281          0.809044          0.760715   
 2            -1.068688         -1.056771         -1.023744         -0.992525   
 3             1.429330          1.510336          1.540319          1.553951   
 4             0.136900          0.173089          0.201047          0.233120   
 ...                ...               ...               ...               ...   
 5057          0.290156          0.268659          0.227433          0.174327   
 5058         -1.863825         -1.793695         -1.698843         -1.591567   
 5059         -1.336976         -1.363791         -1.412324         -1.430077   
 5060          0.511147          0.425766          0.333155          0.236047   
 5061         -0.167306         -0.206049         -0.274168         -0.339079   
 
       1008.594848